# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Clasificación de relaciones con facebook/bart-large-mnli

Model page: https://huggingface.co/facebook/bart-large-mnli

Entailment is calculated in both directions (forward probs (Arg1->Arg2) and backward probs (Arg2->Arg1)), then the label is decided with the following criteria:
   
Bidirectional decision:
  - Rephrase: high entailment both ways, low contradiction
  - Attack: contradiction high either way
  - Support: entailment high at least one way, and not clearly neutral/contradictory
  - No Relationship: otherwise

# No keywords

In [1]:
import os
import re
import math
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gc

process_rel_path = r"/kaggle/working/TFM/Data/Relationships No Keywords"

model_name = "facebook/bart-large-mnli"
device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 50                 
MODEL_TAG = "bart"      
VALID_OUT = {"Support", "Attack", "Rephrase", "No Relationship"}

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

# figure out label 
id2label = {int(k): v.lower() for k, v in model.config.id2label.items()}
label2id = {v: int(k) for k, v in id2label.items()}

print(f'DeBERTa v3 base mnli labels {label2id}')

IDX_ENT = label2id["entailment"]
IDX_CON = label2id["contradiction"]
IDX_NEU = label2id["neutral"]

# Heuristic thresholds 
ENT_THR       = 0.50   
CONTR_THR     = 0.50   
REPHRASE_THR  = 0.75  
MAX_NEUTRAL   = 0.70   

# Margins: “X wins by at least this much”
SUPPORT_MARGIN = 0.10  # p_ent - p_con must exceed this (either direction)
ATTACK_MARGIN  = 0.10  # p_con - p_ent must exceed this (either direction)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

2025-08-25 17:43:18.567781: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756143798.926177      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756143799.033768      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

DeBERTa v3 base mnli labels {'contradiction': 0, 'neutral': 1, 'entailment': 2}


In [2]:
!git clone https://github.com/camipalo/TFM.git

Cloning into 'TFM'...
remote: Enumerating objects: 2510, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 2510 (delta 48), reused 56 (delta 39), pack-reused 2444 (from 2)
Receiving objects: 100% (2510/2510), 109.09 MiB | 12.73 MiB/s, done.
Resolving deltas: 100% (2104/2104), done.
Updating files: 100% (1065/1065), done.


In [3]:
def preprocess_text(s):
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

@torch.no_grad()
def nli_probs(pairs, max_length: int):
    if not pairs:
        return np.zeros((0,3)), np.zeros((0,3))

    a1 = [preprocess_text(x[0]) for x in pairs]
    a2 = [preprocess_text(x[1]) for x in pairs]

    # forward: premise=a1, hypothesis=a2
    enc_f = tokenizer(a1, a2, return_tensors="pt", padding=True, truncation=True,
                      max_length=max_length).to(device)
    logits_f = model(**enc_f).logits
    prob_f = torch.softmax(logits_f, dim=-1).detach().cpu().numpy()

    # backward: premise=a2, hypothesis=a1
    enc_b = tokenizer(a2, a1, return_tensors="pt", padding=True, truncation=True,
                      max_length=max_length).to(device)
    logits_b = model(**enc_b).logits
    prob_b = torch.softmax(logits_b, dim=-1).detach().cpu().numpy()

    return prob_f, prob_b

def decide_label(p_ent_f, p_con_f, p_neu_f, p_ent_b, p_con_b, p_neu_b):
    # 1) Hard guard: if both sides look very neutral, say NR
    if max(p_neu_f, p_neu_b) >= MAX_NEUTRAL:
        return "No Relationship"

    # 2) Rephrase: bi-directional entailment and low contradiction
    if min(p_ent_f, p_ent_b) >= REPHRASE_THR and max(p_con_f, p_con_b) <= (1 - REPHRASE_THR):
        return "Rephrase"

    # 3) Attack: contradiction wins with margin OR strong contradiction
    if (
        (p_con_f - p_ent_f) >= ATTACK_MARGIN or
        (p_con_b - p_ent_b) >= ATTACK_MARGIN or
        p_con_f >= CONTR_THR or
        p_con_b >= CONTR_THR
    ):
        return "Attack"

    # 4) Support: entailment wins with margin OR clears lowered threshold
    if (
        (p_ent_f - p_con_f) >= SUPPORT_MARGIN or
        (p_ent_b - p_con_b) >= SUPPORT_MARGIN or
        p_ent_f >= ENT_THR or
        p_ent_b >= ENT_THR
    ):
        return "Support"

    # 5) Fallback
    return "No Relationship"


def free_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def compute_max_length(input_dir, prefix_substring, safety_limit=512):
    max_len = 0
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    for fn in files:
        path = os.path.join(input_dir, fn)
        try:
            df = pd.read_csv(path)
        except Exception:
            continue
        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            continue
        for a, b in zip(df["SDGarg1"], df["SDGarg2"]):
            a = preprocess_text(str(a) if pd.notna(a) else "")
            b = preprocess_text(str(b) if pd.notna(b) else "")
            ids = tokenizer(a, b, truncation=False, padding=False)["input_ids"]
            max_len = max(max_len, len(ids))
    if max_len == 0:
        max_len = 128
    return min(max_len, safety_limit)

def classify_relationships_deberta(input_dir, prefix_substring, model_tag=MODEL_TAG):
    files = [f for f in os.listdir(input_dir) if f.endswith(".csv") and prefix_substring in f]
    if not files:
        print(f"No CSVs found in '{input_dir}' containing '{prefix_substring}'.")
        return

    MAX_LENGTH = compute_max_length(input_dir, prefix_substring, safety_limit=512)
    print(f"Using MAX_LENGTH = {MAX_LENGTH}")

    rel_col = f"rel_{model_tag}"

    for fn in files:
        path = os.path.join(input_dir, fn)
        print(f"\nProcessing: {path}")
        df = pd.read_csv(path)

        if "SDGarg1" not in df.columns or "SDGarg2" not in df.columns:
            print(f"  Skipped (missing SDGarg1/SDGarg2): {fn}")
            continue

        df[rel_col] = ""

        n = len(df)
        total_done = 0
        first_examples = []

        batches = math.ceil(n / BATCH_SIZE)
        for bi in range(batches):
            s = bi * BATCH_SIZE
            e = min((bi + 1) * BATCH_SIZE, n)
            chunk = df.iloc[s:e]

            pairs = list(zip(chunk["SDGarg1"].astype(str).tolist(),
                             chunk["SDGarg2"].astype(str).tolist()))
            try:
                prob_f, prob_b = nli_probs(pairs, max_length=MAX_LENGTH)
                labels = []
                for i in range(len(pairs)):
                    pf = prob_f[i]; pb = prob_b[i]
                    p_ent_f, p_neu_f, p_con_f = pf[IDX_ENT], pf[IDX_NEU], pf[IDX_CON]
                    p_ent_b, p_neu_b, p_con_b = pb[IDX_ENT], pb[IDX_NEU], pb[IDX_CON]
                    lab = decide_label(p_ent_f, p_con_f, p_neu_f, p_ent_b, p_con_b, p_neu_b)
                    labels.append(lab)
            except Exception as ex:
                print(f"  Batch {bi+1}/{batches} error: {ex}. Marking 'No Relationship'.")
                labels = ["No Relationship"] * len(pairs)

            df.loc[chunk.index, rel_col] = labels

            # first 5 examples
            for (a1, a2), lab in zip(pairs, labels):
                if len(first_examples) < 5:
                    first_examples.append((a1, a2, lab))
                elif len(first_examples) == 5:
                    print("Sample predictions (first 5):\n")
                    for _a1, _a2, _lab in first_examples:
                        print(f"- Arg1: {_a1}\n- Arg2: {_a2}\n  Label: {_lab}\n")
                    first_examples.append((a1, a2, lab))

            total_done += len(pairs)
            if total_done % 100 < BATCH_SIZE:
                print(f"  Progress: {total_done}/{n} relations classified...")

        # Final label validation 
        mask_nan = df["SDGarg1"].isna() | df["SDGarg2"].isna()
        df.loc[mask_nan, rel_col] = "No Relationship"
        bad = ~df[rel_col].isin(VALID_OUT)
        if bad.any():
            df.loc[bad, rel_col] = "No Relationship"

        df.to_csv(path, index=False, encoding="utf-8")
        free_cuda()
        print(f"Saved: {path}")
    return df


## GLOBAL SDG 2023 

#### Qwen2.5 3B extraction

In [4]:
prefix = "intra_goalGLOBAL_SGD2023_qwen2.5-3b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 172

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_qwen2.5-3b.csv


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.58.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Sample predictions (first 5):

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: According to major international studies, few teenagers can differentiate between a fact and an opinion.
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: As the world’s nations prepare to meet in September to review the progress the world has made so far towards achieving the SDGs, at the midpoint of the 2030 Agenda, SDSN emphasizes six areas for immediate action.
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_flanT5,rel_robertaL,rel_deberta,rel_bart
276,The “global financial architecture” (GFA) refe...,Achieving the SDGs requires global cooperation...,0_10,0_22,NaN,No Relationship,No Relationship,Support,No Relationship,No Relationship
482,It is argued since 2017 that a combination of ...,The SDG Index serves as a conversation opener ...,0_26,0_28,NaN,Support,Support,Support,No Relationship,No Relationship
1030,To make sure that existing financial resources...,The most important component of the stimulus p...,17_7,17_8,NaN,Support,Support,Support,No Relationship,No Relationship
114,All UN Member States should adopt long-term su...,The SDG Index serves as a conversation opener ...,0_3,0_28,NaN,No Relationship,No Relationship,Support,No Relationship,No Relationship
208,"In 2022, the United Nations Secretary-General ...",SDG target 4.1 calls for universal access to 1...,0_7,0_20,NaN,No Relationship,Support,Support,No Relationship,No Relationship


rel_bart
No Relationship    1019
Attack               37
Rephrase              2
Name: count, dtype: int64

In [5]:
prefix = "cross_goalGLOBAL_SGD2023_qwen2.5-3b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 180

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_qwen2.5-3b.csv
Sample predictions (first 5):

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: only limited progress is being made on the environmental and biodiversity goals, including SDG 12 (Responsible Consumption and Production), SDG 13 (Climate Action), SDG 14 (Life Below Water), and SDG 15 (Life on Land)
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
- Arg2: And global cooperation has ebbed as geopolitical tensions have risen.
  Label: No Relationship

- Arg1: At their core, the SDGs are an investment agenda: it is critical tha

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_flanT5,rel_robertaL,rel_deberta,rel_bart
8861,ensure universal access to modern energy sources,The most important component of the stimulus p...,7_8,17_8,NaN,Support,Support,No Relationship,No Relationship,No Relationship
11519,"The United States, as the world’s biggest econ...",Governments are only now learning how to desig...,13_5,17_10,NaN,Support,No Relationship,Support,No Relationship,No Relationship
2727,The continuing efforts of the SDSN is a testam...,"Provincial, metropolitan, and city governments...",0_13,12_5,NaN,Support,Support,Support,No Relationship,No Relationship
6060,The greatest responsibility for achieving the ...,Sustainable cities: urban infrastructure and s...,3_2,11_0,NaN,No Relationship,Support,Support,No Relationship,No Relationship
9804,The world has made some progress in strengthen...,"To achieve the SDGs, the world must invest bol...",9_3,17_3,NaN,Support,Support,Support,No Relationship,No Relationship


rel_bart
No Relationship    11267
Attack               467
Rephrase              78
Support               10
Name: count, dtype: int64

#### Gemma3 27B extraction

In [6]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-27b"   
rel_col = f"rel_{model_name}"

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 216

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_gemma3-27b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Despite this alarming development, the SDGs are still achievable.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Nat

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_flanT5,rel_robertaL,rel_deberta,rel_bart
2666,"Unless the SDGs are actively pursued, geophysi...","The Climate Action Tracker, an independent sci...",13_10,13_22,NaN,Support,No Relationship,Support,No Relationship,No Relationship
1782,1. Universal quality education and innovation-...,Countries must further expand and transform ed...,4_0,4_9,NaN,Support,Support,Rephrase,No Relationship,No Relationship
3340,We commend global leaders who “oppose the use ...,The ESDR has served as a conversation-opener w...,16_14,16_25,NaN,No Relationship,Support,No Relationship,No Relationship,No Relationship
3888,Achieving the SDGs will require a transformati...,"Second, developed countries are not being held...",17_15,17_16,NaN,Attack,No Relationship,No Relationship,No Relationship,No Relationship
454,All UN Member States should adopt long-term su...,SDSN has recommended six inter-related long-te...,0_9,0_32,NaN,Support,Support,Support,No Relationship,No Relationship


rel_bart
No Relationship    4077
Attack              106
Support               4
Rephrase              1
Name: count, dtype: int64

In [7]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-27b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 262

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_gemma3-27b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: As called for by United Nations Secretary-General António Guterres, the SDG Stimulus plan has five main components:
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Greatly increased funding for national and subnational governments and private businesses in the emerging economies, especially the low-income countries (LICs) and lower-middle-income countries (LMICs), to carry out needed SDG actions;
  L

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_flanT5,rel_robertaL,rel_deberta,rel_bart
41378,Local governments have the front-line responsi...,There is no hope for global peace unless there...,11_3,16_7,NaN,No Relationship,Support,No Relationship,No Relationship,Attack
28575,SDSN is working closely with the UNESCO SDG 4 ...,Neighboring countries share ecosystems (rivers...,4_5,14_3,NaN,No Relationship,Support,No Relationship,No Relationship,No Relationship
46922,Questions explored policy measures to address ...,The urgent objective of the SDG Stimulus is to...,13_25,17_20,NaN,Support,Support,Support,No Relationship,Attack
637,International cooperation is trapped by bureau...,"Similarly, extreme poverty can lead to a colla...",0_30,1_7,NaN,Attack,Attack,No Relationship,No Relationship,No Relationship
28592,Governments are only now mapping out pathways ...,The SDG Index and Dashboards also include unof...,4_7,14_6,NaN,No Relationship,No Relationship,No Relationship,No Relationship,No Relationship


rel_bart
No Relationship    48235
Attack              1766
Rephrase              79
Support               17
Name: count, dtype: int64

#### Gemma3 4B extraction

In [8]:
prefix = "intra_goalGLOBAL_SGD2023_gemma3-4b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 262

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_gemma3-4b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: None of their objectives are beyond our reach.
  Label: Attack

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The SDGs are still achievable.
  Label: Attack

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: It is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
  Label: No Relationship

- Arg1: At the midpoint of the 203

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_deberta,rel_bart
3506,"Creation of ambitious, internationally-agreed ...",Success requires a combination of better metri...,0_22,0_108,NaN,Support,Support,No Relationship,No Relationship
20034,The world is also seriously off track to meet ...,the SDG Index has included carbon dioxide emis...,13_0,13_33,NaN,No Relationship,Support,No Relationship,No Relationship
2608,"UN Member States should adopt an SDG Stimulus,...",Governments are only now mapping out pathways ...,0_16,0_89,NaN,Support,No Relationship,No Relationship,No Relationship
1715,"All countries, poorer and richer alike, should...",This assessment of government commitment and e...,0_10,0_111,NaN,Support,Support,No Relationship,No Relationship
1329,Align private business investment flows with t...,"Unless the SDGs are actively pursued, geophysi...",0_8,0_38,NaN,Support,Support,No Relationship,No Relationship


rel_bart
No Relationship    22275
Attack               624
Rephrase              11
Support                4
Name: count, dtype: int64

In [9]:
prefix = "cross_goalGLOBAL_SGD2023_gemma3-4b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 262

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_gemma3-4b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The world is off track, but that is all the more reason to double down on the SDGs.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: It is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Revise liquidity structures for LICs and LMICs, especially regarding sovereign debts, to forestall self-fulfilling banking and balance-of-payments crises.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Create 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_deberta,rel_bart
75176,"The collaboration with SDSN is ongoing, with t...",Protection and sustainable management of rainf...,0_159,15_11,NaN,Support,No Relationship,No Relationship,No Relationship
213976,the SDGs represent an investment agenda: to de...,Reform current institutional frameworks and de...,12_14,16_17,NaN,Support,Support,No Relationship,Attack
149019,The UN General Assembly’s Transforming Educati...,acidification of the oceans (with an increase ...,4_6,14_5,NaN,No Relationship,No Relationship,No Relationship,No Relationship
96654,Nor did countries have a common language to di...,"all countries, richer and poorer alike, should...",1_46,6_21,NaN,Support,Support,No Relationship,No Relationship
194201,The GFA also requires alignment of the private...,explosions of invasive marine species due to i...,9_22,14_8,NaN,No Relationship,No Relationship,No Relationship,No Relationship


rel_bart
No Relationship    217987
Attack               7297
Rephrase              557
Support               110
Name: count, dtype: int64

#### Llama3.3 70B extraction

In [10]:
prefix = "intra_goalGLOBAL_SGD2023_llama3.3-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 250

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_llama3.3-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Since the outbreak of the pandemic in 2020 and other simultaneous crises, SDG progress has stalled globally.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The world is off track, but that is all the more reason to double down on the SDGs.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_llama,rel_flanT5,rel_robertaL,rel_deberta,rel_bart
3899,The SDG policy agenda is complex. The SDGs cal...,"The key finding from this survey is that, seve...",0_45,0_75,NaN,Support,Support,Support,No Relationship,No Relationship
3976,SDSN’s strategy to achieve the SDGs,International financing flows should be aligne...,0_46,0_90,NaN,Support,Support,Support,No Relationship,No Relationship
10727,National governments must ensure both the dome...,"Argentina, Barbados, Chile, Germany, Jamaica a...",17_16,17_59,NaN,Support,No Relationship,Support,No Relationship,No Relationship
4678,The SDG Index and Dashboards track the annual ...,when only 57 percent of surveyed countries had...,0_59,0_77,NaN,No Relationship,No Relationship,Rephrase,No Relationship,No Relationship
7159,This metric is particularly useful for assessi...,cities are where the climate battle will large...,11_12,11_15,NaN,No Relationship,No Relationship,Support,No Relationship,No Relationship


rel_bart
No Relationship    11728
Attack               275
Support                6
Rephrase               2
Name: count, dtype: int64

In [11]:
prefix = "cross_goalGLOBAL_SGD2023_llama3.3-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

/tmp/ipykernel_19/2617973462.py:72: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Using MAX_LENGTH = 304

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_llama3.3-70b.csv


/tmp/ipykernel_19/2617973462.py:100: DtypeWarning: Columns (7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track.
  Label: Support

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Similarly, extreme poverty can lead to a collapse of tax revenues, followed by government bankruptcy and further economic collapse, a syndrome that now threatens dozens of poor countries.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: In 2022, Investment per person in the LICs averaged a meagre US$175 per person, compared with 

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_flanT5,rel_robertaL,rel_llama,rel_deberta,rel_bart
52367,"In HICs and LICs, the pandemic and other crise...",acidification of the oceans (with an increase ...,3_10,13_11,NaN,No Relationship,No Relationship,NaN,NaN,No Relationship
101056,The Ongoing Conflict Score builds on six indic...,"Recently, China has reiterated its support for...",16_31,17_17,NaN,Support,No Relationship,NaN,NaN,No Relationship
128,Further investment is needed in statistical ca...,"At the global level, averaging across countrie...",0_8,1_0,NaN,No Relationship,Support,Attack,No Relationship,No Relationship
80648,It is also vital to share fairly and globally ...,Almost all governments have committed to adopt...,10_9,17_45,NaN,Support,Support,NaN,NaN,No Relationship
39313,"Comments submitted by governments, researchers...","Recently, China has reiterated its support for...",0_97,17_17,NaN,Support,Support,NaN,NaN,No Relationship


rel_bart
No Relationship    98082
Attack              3329
Rephrase              82
Support               22
Name: count, dtype: int64

#### Deepseek r1 70B extraction

In [12]:
prefix = "intra_goalGLOBAL_SGD2023_deepseek-r1-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

Using MAX_LENGTH = 368

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/intra_goalGLOBAL_SGD2023_deepseek-r1-70b.csv
Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: Despite this alarming development, the SDGs are still achievable. None of their objectives are beyond our reach.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The world is off track, but that is all the more reason to double down on the SDGs.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At their core, the SDGs are an investment agenda: it is critical that UN Member States adopt and implement the SDG Stimulus and support a comprehensive reform of the global financial architecture.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
-

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_robertaL,rel_llama,rel_flanT5,rel_deberta,rel_bart
12647,Dire shortfalls in meeting the SDGs to reachin...,the slowdown of ocean circulation due to clima...,13_4,13_9,NaN,Rephrase,Attack,No Relationship,No Relationship,No Relationship
16047,"For example, globalized trade rules for ‘clean...",One of the consistent findings of the SDSN is ...,17_30,17_46,NaN,Support,Support,No Relationship,No Relationship,No Relationship
1766,All UN Member States and UN agencies can count...,We emphasize that all these global goals are i...,0_13,0_25,NaN,Support,Support,Support,No Relationship,No Relationship
1348,"At the halfway mark to 2030, there remains a g...",The SDG Index relies on inputs from the SDSN n...,0_9,0_125,NaN,Support,No Relationship,Support,No Relationship,No Relationship
12238,Local governments have the front-line responsi...,The dashboards highlight persisting gaps betwe...,11_4,11_9,NaN,Support,Support,No Relationship,No Relationship,No Relationship


rel_bart
No Relationship    16802
Attack               297
Rephrase              10
Support                4
Name: count, dtype: int64

In [13]:
prefix = "cross_goalGLOBAL_SGD2023_deepseek-r1-70b"   

args_classified = classify_relationships_deberta(process_rel_path, prefix)
display(args_classified.sample(5))

args_classified["rel_bart"].value_counts()

/tmp/ipykernel_19/2617973462.py:72: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Using MAX_LENGTH = 393

Processing: /kaggle/working/TFM/Data/Relationships No Keywords/cross_goalGLOBAL_SGD2023_deepseek-r1-70b.csv


/tmp/ipykernel_19/2617973462.py:100: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Sample predictions (first 5):

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: To achieve the SDGs the world must both alter its current investment patterns and increase the overall volume of investments.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: The grim reality is that at the midpoint of the 2030 Agenda, the SDGs are far off track.
  Label: Support

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: At the global level, averaging across countries, not a single SDG is currently projected to be met by 2030, with the poorest countries struggling the most.
  Label: No Relationship

- Arg1: At the midpoint of the 2030 Agenda, all of the SDGs are seriously off track.
- Arg2: 1. Increased funding from the multilateral develop-ment banks (MDBs) and public development banks (PDBs) to low- and middle-income countries, linked to inve

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,SDGarg1,SDGarg2,id_arg1,id_arg2,rel,rel_robertaL,rel_flanT5,rel_llama,rel_bart
53276,All UN Member States should present VNRs at le...,Many civil-society and private-sector actors h...,0_75,17_73,NaN,Support,Support,No Relationship,No Relationship
107268,"1. Deep, chronic, and crippling under-investme...",The OECD estimates that 105 of the 169 SDG tar...,9_3,11_10,NaN,Support,No Relationship,NaN,No Relationship
26360,The 2030 Agenda and the SDGs recognize the imp...,"In her 2022 report on the SDGs, E. Tendayi Ach...",0_92,10_4,NaN,Support,Support,Support,No Relationship
67109,"According to IMF estimates in 2019, the financ...",Social justice and sustainable development req...,1_17,16_4,NaN,No Relationship,Support,NaN,No Relationship
116505,Disadvantaged communities have lower access to...,All UN Member States should recommit to peacef...,10_17,17_14,NaN,No Relationship,Support,NaN,No Relationship


rel_bart
No Relationship    115000
Attack               3130
Rephrase              107
Support                22
Name: count, dtype: int64